> **SUPERSEDED - Earth Engine already serves MERRA-2 hourly (cadence 1 h), so no
> NASA GES DISC download is needed. Build UTC-6 days with
> MERRA2_Square_Export_Colab.ipynb and DAY_MODE = 'UTC-6'.**

# MERRA-2 hourly download and UTC-6 daily stacks

Companion to `MERRA2_Square_Export_Colab.ipynb` - same layout, same band naming
(`YYYYMMDD_<VAR>`), same "check before you download" discipline.

The difference is the source. That notebook used Earth Engine's **daily**
MERRA-2 products, which are means over the **UTC day**. The PM2.5 target is a
mean over the **UTC-6 day** (fixed CST, no DST):

```
UTC-6 day D  =  [D 06:00 UTC, D+1 06:00 UTC)
```

A daily mean cannot be re-centred by six hours, so this notebook goes back to
the **hourly** `tavg1_2d_*` collections, downloads a subset, and builds the
UTC-6 stacks itself. The hourly files also make other daily statistics free
(PBLH max/min, ...).

## Setup (once)

1. Register at <https://urs.earthdata.nasa.gov>.
2. Under *Applications*, approve **NASA GESDISC DATA ARCHIVE**.
3. `earthaccess.login()` below prompts for the username and password. Nothing
   is written into this notebook or to disk.

## What the data looks like

The granules are hourly: 24 records per file, stamped `00:30 ... 23:30`,
each the mean of `[HH:00, HH+1:00)`. Granule names carry the processing
stream, and CMR confirms it varies:

| period | stream |
|---|---|
| 2010 | `MERRA2_300` |
| 2011 onward | `MERRA2_400` |
| reprocessed months (e.g. 2020-09) | `MERRA2_401` |

So the name is always discovered through CMR, never assembled by hand.

## Outputs

```
hourly/MERRA2_{AER,FLX,SLV}_hourly_YYYYMMDD.nc4
utc6_daily/merra2_{aer,flx,slv}_utc6_daily_YYYY_MM.tif
```


In [ ]:
!pip install -q earthaccess xarray netCDF4 rasterio requests

In [ ]:
import datetime as dt
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import earthaccess
import numpy as np
import pandas as pd
import rasterio
import requests
import xarray as xr
from rasterio.transform import from_origin

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('earthaccess', earthaccess.__version__, '| Colab:', IN_COLAB)

In [ ]:
# ---- configuration -------------------------------------------------------
ROOT = Path('/content/merra2')        # point at a Drive path to survive the session
RAW  = ROOT / 'hourly'
PROC = ROOT / 'utc6_daily'
for p in (RAW, PROC):
    p.mkdir(parents=True, exist_ok=True)

START = dt.date(2010, 1, 1)
END   = dt.date(2024, 12, 31)         # the last UTC-6 day also needs 2025-01-01's first 6 h

# Study square plus margin. The native grid is lon = -180 + 0.625 i,
# lat = -90 + 0.5 j, but the index ranges are taken from the file, not the formula.
LON_RANGE = (-100.3, -65.2)
LAT_RANGE = (37.0, 60.8)

VERSION = '5.12.4'
CMR = 'https://cmr.earthdata.nasa.gov/search/granules.json'

COLLECTIONS = {
    'AER': dict(short='M2T1NXAER',
                variables=['BCCMASS', 'OCCMASS', 'SO4CMASS', 'SSCMASS25', 'DUCMASS25',
                           'BCSMASS', 'OCSMASS', 'SO4SMASS', 'SSSMASS25', 'DUSMASS25',
                           'BCEXTTAU', 'OCEXTTAU', 'SUEXTTAU', 'DUEXTTAU', 'TOTEXTTAU']),
    'FLX': dict(short='M2T1NXFLX',
                variables=['PBLH', 'QLML', 'TLML', 'ULML', 'VLML', 'SPEED', 'SPEEDMAX']),
    'SLV': dict(short='M2T1NXSLV',
                variables=['U2M', 'V2M', 'U10M', 'V10M', 'U50M', 'V50M',
                           'U250', 'V250', 'U500', 'V500', 'U850', 'V850']),
}

# Empty by default. Candidates: SLV T2M, QV2M, PS, SLP, TQV; FLX PRECTOT.
EXTRA_VARIABLES = {}
for _k, _v in EXTRA_VARIABLES.items():
    COLLECTIONS[_k]['variables'] += list(_v)

# mean is always built; these add <VAR>_MAX / <VAR>_MIN bands from the same hours
EXTRA_STATS = {'PBLH': ['max', 'min'], 'TLML': ['max', 'min']}

WORKERS = 4
MAX_RETRIES = 5

print('days:', (END - START).days + 1, '| collections:', list(COLLECTIONS))

In [ ]:
# ---- authentication ------------------------------------------------------
earthaccess.login()                              # prompts on Colab
SESSION = earthaccess.get_requests_https_session()
print('session ready')

In [ ]:
# ---- granule index from CMR (no download yet) ---------------------------
# One query per collection per year instead of one per day: ~45 calls total
# instead of 16,440. Returns {YYYYMMDD: (granule_name, opendap_url)}.
def cmr_index(short, start, end, session=SESSION, version=VERSION):
    idx = {}
    t = start
    while t <= end:
        q = dict(short_name=short, version=version,
                 temporal=f'{t},{min(t + dt.timedelta(days=365), end)}',
                 page_size=2000, sort_key='start_date')
        r = session.get(CMR, params=q, timeout=180)
        r.raise_for_status()
        for g in r.json()['feed']['entry']:
            name = g.get('producer_granule_id') or g.get('title')
            day = name.split('.')[2]                      # YYYYMMDD
            op = [l['href'] for l in g.get('links', [])
                  if 'opendap' in (l.get('href') or '').lower()]
            if op:
                idx[day] = (name, op[0])
        t += dt.timedelta(days=365)
    return idx

INDEX = {}
for key, spec in COLLECTIONS.items():
    INDEX[key] = cmr_index(spec['short'], START, END + dt.timedelta(days=1))
    streams = sorted({v[0].split('.')[0] for v in INDEX[key].values()})
    print(f'{key}: {len(INDEX[key])} granules, streams {streams}')

print('\nexample OPeNDAP URL (this is the service CMR advertises today):')
print(' ', INDEX['AER'][START.strftime('%Y%m%d')][1])

In [ ]:
# ---- variable check + coordinate read, one granule ----------------------
# The .dds is the cheap text listing of variables and dimensions.
probe_day = dt.date(2015, 7, 1).strftime('%Y%m%d')
probe_url = INDEX['AER'][probe_day][1]
print('granule:', INDEX['AER'][probe_day][0])
print('url    :', probe_url)

r = SESSION.get(probe_url + '.dds', timeout=120)
r.raise_for_status()
missing = [v for v in COLLECTIONS['AER']['variables'] if v not in r.text]
print('DDS bytes:', len(r.text), '| missing:', missing if missing else 'none')

# coordinate arrays, so the index ranges come from the file itself
rc = SESSION.get(probe_url + '.ascii?lat,lon', timeout=120)
rc.raise_for_status()
print(rc.text[:400])

In [ ]:
# ---- constraint expression and one-granule subset -----------------------
# Read lat/lon by fetching the two coordinate variables as NetCDF first.
import io

def opendap_subset(url, variables, j0, j1, i0, i1):
    ce = ','.join([f'{v}[0:23][{j0}:{j1}][{i0}:{i1}]' for v in variables]
                  + ['lat', 'lon', 'time'])
    return url + '?' + ce

def open_subset(url, ce):
    """Download the subset through the authenticated session, then open locally.
    xarray cannot reuse our Earthdata session, so we never open the URL directly."""
    r = SESSION.get(url + ce, timeout=600)
    r.raise_for_status()
    return xr.open_dataset(io.BytesIO(r.content), engine='netcdf4')

coords = open_subset(probe_url, '?lat,lon')
lat, lon = coords['lat'].values, coords['lon'].values
j0, j1 = int(np.searchsorted(lat, LAT_RANGE[0])), int(np.searchsorted(lat, LAT_RANGE[1])) - 1
i0, i1 = int(np.searchsorted(lon, LON_RANGE[0])), int(np.searchsorted(lon, LON_RANGE[1])) - 1
print(f'lat {lat[j0]:.2f}..{lat[j1]:.2f}  j {j0}..{j1}  ({j1-j0+1} px)')
print(f'lon {lon[i0]:.2f}..{lon[i1]:.2f}  i {i0}..{i1}  ({i1-i0+1} px)')

CE_AER = opendap_subset('', COLLECTIONS['AER']['variables'], j0, j1, i0, i1)
print('\nconstraint expression (truncated):')
print(CE_AER[:300], '...')

t0 = time.time()
ds = open_subset(probe_url, CE_AER)
print(f'\nsubset fetched in {time.time()-t0:.1f} s')
print('dims:', dict(ds.sizes))
print('time:', str(ds.time.values[0]), '..', str(ds.time.values[-1]))
assert ds.sizes['time'] == 24, 'expected 24 hourly steps'
assert ds.sizes['lat'] == j1 - j0 + 1 and ds.sizes['lon'] == i1 - i0 + 1, 'CE was not honoured'
print('variables:', [v for v in COLLECTIONS['AER']['variables'] if v in ds.variables])

In [ ]:
# ---- timed test on one month, all three collections ----------------------
import calendar

def raw_path(key, day):
    return RAW / f'MERRA2_{key}_hourly_{day:%Y%m%d}.nc4'

def fetch_day(key, day, j0, j1, i0, i1):
    """Subset one UTC day, validate, write atomically. Returns (path, status, bytes)."""
    path = raw_path(key, day)
    if path.exists():
        return path, 'exists', path.stat().st_size
    key_day = day.strftime('%Y%m%d')
    if key_day not in INDEX[key]:
        raise RuntimeError(f'no granule for {key} {day}')
    name, url = INDEX[key][key_day]
    ce = opendap_subset('', COLLECTIONS[key]['variables'], j0, j1, i0, i1)
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            r = SESSION.get(url + ce, timeout=900)
            if r.status_code in (429, 500, 502, 503, 504):
                raise IOError(f'HTTP {r.status_code}')
            r.raise_for_status()
            tmp = path.with_suffix('.part')
            tmp.write_bytes(r.content)
            tmp.replace(path)
            return path, 'ok', len(r.content)
        except Exception as exc:
            last = exc
            time.sleep(2 ** attempt)
    raise RuntimeError(str(last)[:160])

test_month = (2010, 1)
n_days = calendar.monthrange(*test_month)[1]
t0 = time.time()
sizes = []
for key in COLLECTIONS:
    for d in range(1, n_days + 1):
        _, _, size = fetch_day(key, dt.date(*test_month, d), j0, j1, i0, i1)
        sizes.append(size)
elapsed = time.time() - t0

per_req = elapsed / (n_days * len(COLLECTIONS))
total_req = ((END - START).days + 2) * len(COLLECTIONS)
hours, gb = total_req * per_req / 3600, total_req * float(np.mean(sizes)) / 1e9
print(f'{n_days*len(COLLECTIONS)} requests in {elapsed/60:.1f} min '
      f'({per_req:.1f} s each, mean {np.mean(sizes)/1e6:.1f} MB)')
print(f'projected full run: {hours:.1f} h, {gb:.1f} GB, {total_req} requests')
if hours > 12 or gb > 60:
    print('*** over the 12 h / 60 GB budget - stop and reconsider ***')

In [ ]:
# ---- full download -------------------------------------------------------
# Resumable: existing files are skipped, so this cell is safe to interrupt.
days = [START + dt.timedelta(days=i) for i in range((END - START).days + 2)]
todo = [(k, d) for k in COLLECTIONS for d in days if not raw_path(k, d).exists()]
print(f'{len(todo)} to fetch of {len(days)*len(COLLECTIONS)}')

errors = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futs = {pool.submit(fetch_day, k, d, j0, j1, i0, i1): (k, d) for k, d in todo}
    for n, fut in enumerate(as_completed(futs), 1):
        k, d = futs[fut]
        try:
            fut.result()
        except Exception as exc:
            errors.append((k, d.isoformat(), str(exc)[:120]))
        if n % 200 == 0:
            print(f'  {n}/{len(todo)}  errors={len(errors)}')
print('failures:', len(errors))
if errors:
    pd.DataFrame(errors, columns=['collection', 'date', 'error']).to_csv(
        ROOT / 'failed_downloads.csv', index=False)

In [ ]:
# ---- validation summary --------------------------------------------------
def validate(key, path):
    try:
        with xr.open_dataset(path) as ds:
            ok = (ds.sizes.get('time') == 24
                  and all(v in ds.variables for v in COLLECTIONS[key]['variables'])
                  and ds.sizes.get('lat') == j1 - j0 + 1
                  and ds.sizes.get('lon') == i1 - i0 + 1)
            return ok, ds.sizes.get('time')
    except Exception as exc:
        return False, str(exc)[:60]

rows = []
for key in COLLECTIONS:
    for d in days:
        p = raw_path(key, d)
        ok, info = validate(key, p) if p.exists() else (False, 'missing')
        rows.append(dict(collection=key, date=d.isoformat(), exists=p.exists(),
                         valid=ok, detail=str(info)))
summary = pd.DataFrame(rows)
print(summary.groupby('collection')[['exists', 'valid']].sum().to_string())
print('invalid but present:', int((summary.exists & ~summary.valid).sum()))

In [ ]:
# ---- build UTC-6 daily stacks -------------------------------------------
# Day D uses the 24 half-hour stamps D 06:30 .. D+1 05:30. Any missing hour
# makes that day-variable NaN; we never average fewer than 24 hours.
def utc6_hours(key, day):
    f0, f1 = raw_path(key, day), raw_path(key, day + dt.timedelta(days=1))
    if not (f0.exists() and f1.exists()):
        return None
    nxt = day + dt.timedelta(days=1)
    with xr.open_dataset(f0) as a, xr.open_dataset(f1) as b:
        late = a.sel(time=slice(f'{day} 06:30', f'{day} 23:30'))
        early = b.sel(time=slice(f'{nxt} 00:30', f'{nxt} 05:30'))
        if late.sizes['time'] + early.sizes['time'] != 24:
            return None
        return xr.concat([late, early], dim='time').load()

def month_tif(key, year, month):
    out = PROC / f'merra2_{key.lower()}_utc6_daily_{year}_{month:02d}.tif'
    nd = calendar.monthrange(year, month)[1]
    bands, names = [], []
    latv = lonv = None
    for d in range(1, nd + 1):
        day = dt.date(year, month, d)
        chunk = utc6_hours(key, day)
        tag = day.strftime('%Y%m%d')
        if chunk is not None:
            latv, lonv = chunk['lat'].values, chunk['lon'].values
        for v in COLLECTIONS[key]['variables']:
            if chunk is None or v not in chunk:
                arr = np.full((j1 - j0 + 1, i1 - i0 + 1), np.nan, 'float32')
            else:
                arr = chunk[v].mean('time').values.astype('float32')
            bands.append(np.flipud(arr)); names.append(f'{tag}_{v}')
        for v, stats in EXTRA_STATS.items():
            if v not in COLLECTIONS[key]['variables']:
                continue
            for st in stats:
                arr = (getattr(chunk[v], st)('time').values.astype('float32')
                       if chunk is not None else np.full((j1 - j0 + 1, i1 - i0 + 1), np.nan, 'float32'))
                bands.append(np.flipud(arr)); names.append(f'{tag}_{v}_{st.upper()}')
    if latv is None:
        return None
    transform = from_origin(lonv.min() - 0.3125, latv.max() + 0.25, 0.625, 0.5)
    with rasterio.open(out, 'w', driver='GTiff', width=lonv.size, height=latv.size,
                       count=len(bands), dtype='float32', crs='EPSG:4326',
                       transform=transform, nodata=np.nan, compress='deflate') as dst:
        for i, (arr, nm) in enumerate(zip(bands, names), 1):
            dst.write(arr, i)
            dst.set_band_description(i, nm)
    return out

written = []
for key in COLLECTIONS:
    for y in range(START.year, END.year + 1):
        for m in range(1, 13):
            p = month_tif(key, y, m)
            if p:
                written.append(p)
print('monthly stacks:', len(written))
print('example:', written[0].name if written else None)

## Checks

- completeness: every day present and valid, per collection;
- UTC-6 day D equals the mean of D 06:30 .. D+1 05:30, recomputed independently;
- UTC-6 differs from the UTC-day mean for the same date;
- plausible ranges, and the units the source reports.

The comparison against the Earth Engine exports needs their zip archives, which
live on the Windows machine, so that check belongs to `code/check_merra_exports.py`.


In [ ]:
# ---- check 1: completeness ----------------------------------------------
bad = summary[~summary.valid]
print('expected files:', len(days) * len(COLLECTIONS),
      '| invalid:', len(bad))
if len(bad):
    print(bad.groupby('collection').size().to_string())

# ---- check 2: UTC-6 equals an independent recomputation ------------------
for day in [dt.date(2015, 7, 1), dt.date(2018, 1, 15), dt.date(2022, 8, 20)]:
    for key in ('AER', 'FLX'):
        chunk = utc6_hours(key, day)
        if chunk is None:
            print(f'{key} {day}: missing hours'); continue
        v = COLLECTIONS[key]['variables'][0]
        pooled = chunk[v].values.reshape(24, -1)[:, 0]
        print(f'{key} {day} {v}: ours={pooled.mean():.6g} '
              f'manual={np.mean(pooled):.6g} n={pooled.size}')

# ---- check 3: UTC-6 vs UTC-day ------------------------------------------
day = dt.date(2015, 7, 1)
with xr.open_dataset(raw_path('FLX', day)) as a:
    utc = float(a['PBLH'].mean('time').values.ravel()[0])
c6 = utc6_hours('FLX', day)
print(f'PBLH {day}: UTC-day {utc:.2f} m vs UTC-6 {float(c6["PBLH"].mean("time").values.ravel()[0]):.2f} m')

# ---- check 4: plausibility ----------------------------------------------
for key, v, lo, hi in [('FLX', 'PBLH', 0, 4000), ('FLX', 'TLML', 200, 330),
                       ('AER', 'BCSMASS', 0, 1e-6)]:
    p = raw_path(key, day)
    with xr.open_dataset(p) as ds:
        a = ds[v].values
        print(f'{key} {v}: {np.nanmin(a):.3g} .. {np.nanmax(a):.3g} '
              f'{ds[v].attrs.get("units", "?")} '
              f'{"OK" if lo <= np.nanmin(a) and np.nanmax(a) <= hi else "CHECK"}')